# Q12: Implement Backpropagation and Gradient Descent (2-Layer ANN)
**Topic:** Coding-question | **Difficulty:** Medium | **Time:** 30 min
**Sheet:** Grind75ML

---

## Question
Implement a 2-layer neural network from scratch with forward pass, backpropagation, and gradient descent training.

## Theory

A 2-layer ANN (1 hidden layer + 1 output layer):

**Forward Pass:**
1. $z_1 = XW_1 + b_1$
2. $a_1 = \sigma(z_1)$ (activation, e.g., ReLU or sigmoid)
3. $z_2 = a_1 W_2 + b_2$
4. $\hat{y} = \sigma(z_2)$ (output activation)

**Backpropagation** (chain rule):
1. $\delta_2 = \hat{y} - y$ (output error)
2. $\frac{\partial L}{\partial W_2} = a_1^T \cdot \delta_2$
3. $\delta_1 = \delta_2 \cdot W_2^T \odot \sigma'(z_1)$ (hidden error)
4. $\frac{\partial L}{\partial W_1} = X^T \cdot \delta_1$

**Gradient Descent Update:**
$$W := W - \alpha \cdot \frac{\partial L}{\partial W}$$

In [ ]:
import numpy as np
import matplotlib.pyplot as plt


class TwoLayerNN:
    """A simple 2-layer neural network for binary classification.
    
    Architecture: Input -> Hidden (sigmoid) -> Output (sigmoid)
    Loss: Binary Cross-Entropy
    """
    
    def __init__(self, input_dim: int, hidden_dim: int, output_dim: int, lr: float = 0.1):
        self.lr = lr
        # Xavier initialization
        self.W1 = np.random.randn(input_dim, hidden_dim) * np.sqrt(2.0 / input_dim)
        self.b1 = np.zeros((1, hidden_dim))
        self.W2 = np.random.randn(hidden_dim, output_dim) * np.sqrt(2.0 / hidden_dim)
        self.b2 = np.zeros((1, output_dim))
        self.loss_history: list[float] = []
    
    @staticmethod
    def sigmoid(z: np.ndarray) -> np.ndarray:
        return 1 / (1 + np.exp(-np.clip(z, -500, 500)))
    
    @staticmethod
    def sigmoid_derivative(a: np.ndarray) -> np.ndarray:
        """Derivative of sigmoid given the sigmoid output a."""
        return a * (1 - a)
    
    def forward(self, X: np.ndarray) -> tuple:
        """Forward pass through the network."""
        z1 = X @ self.W1 + self.b1
        a1 = self.sigmoid(z1)
        z2 = a1 @ self.W2 + self.b2
        a2 = self.sigmoid(z2)
        return z1, a1, z2, a2
    
    def compute_loss(self, y: np.ndarray, y_pred: np.ndarray, eps: float = 1e-12) -> float:
        """Binary cross-entropy loss."""
        y_pred = np.clip(y_pred, eps, 1 - eps)
        return -np.mean(y * np.log(y_pred) + (1 - y) * np.log(1 - y_pred))
    
    def backward(self, X: np.ndarray, y: np.ndarray, z1, a1, z2, a2) -> None:
        """Backpropagation: compute gradients and update weights."""
        n = X.shape[0]
        
        # Output layer gradients
        delta2 = a2 - y  # dL/dz2 for BCE + sigmoid
        dW2 = (1 / n) * (a1.T @ delta2)
        db2 = (1 / n) * np.sum(delta2, axis=0, keepdims=True)
        
        # Hidden layer gradients (chain rule)
        delta1 = (delta2 @ self.W2.T) * self.sigmoid_derivative(a1)
        dW1 = (1 / n) * (X.T @ delta1)
        db1 = (1 / n) * np.sum(delta1, axis=0, keepdims=True)
        
        # Gradient descent update
        self.W2 -= self.lr * dW2
        self.b2 -= self.lr * db2
        self.W1 -= self.lr * dW1
        self.b1 -= self.lr * db1
    
    def train(self, X: np.ndarray, y: np.ndarray, epochs: int = 1000) -> 'TwoLayerNN':
        """Train the network."""
        for epoch in range(epochs):
            z1, a1, z2, a2 = self.forward(X)
            loss = self.compute_loss(y, a2)
            self.loss_history.append(loss)
            self.backward(X, y, z1, a1, z2, a2)
            if (epoch + 1) % 200 == 0:
                print(f"Epoch {epoch+1:4d} | Loss: {loss:.6f}")
        return self
    
    def predict(self, X: np.ndarray) -> np.ndarray:
        _, _, _, a2 = self.forward(X)
        return (a2 >= 0.5).astype(int)

In [ ]:
# --- XOR Problem (non-linearly separable) ---
np.random.seed(42)

X = np.array([[0, 0], [0, 1], [1, 0], [1, 1]], dtype=float)
y = np.array([[0], [1], [1], [0]], dtype=float)  # XOR

nn = TwoLayerNN(input_dim=2, hidden_dim=4, output_dim=1, lr=2.0)
nn.train(X, y, epochs=2000)

# Test predictions
print("\n=== Predictions ===")
_, _, _, probs = nn.forward(X)
preds = nn.predict(X)
for i in range(len(X)):
    print(f"Input: {X[i]} -> Prob: {probs[i][0]:.4f} -> Pred: {preds[i][0]} (True: {int(y[i][0])})")

In [ ]:
# --- Visualize training ---
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Loss curve
axes[0].plot(nn.loss_history)
axes[0].set_title('Training Loss (Binary Cross-Entropy)')
axes[0].set_xlabel('Epoch')
axes[0].set_ylabel('Loss')

# Decision boundary
xx, yy = np.meshgrid(np.linspace(-0.5, 1.5, 200), np.linspace(-0.5, 1.5, 200))
grid = np.c_[xx.ravel(), yy.ravel()]
_, _, _, Z = nn.forward(grid)
Z = Z.reshape(xx.shape)
axes[1].contourf(xx, yy, Z, levels=50, cmap='RdBu_r', alpha=0.8)
axes[1].scatter(X[:, 0], X[:, 1], c=y.flatten(), cmap='RdBu_r', edgecolors='black', s=100)
axes[1].set_title('Decision Boundary (XOR Problem)')

plt.tight_layout()
plt.show()

## Key Takeaways
- **Backpropagation = chain rule** applied layer by layer from output to input
- XOR is the classic test — it's NOT linearly separable, proving why we need hidden layers
- **Xavier initialization** prevents vanishing/exploding gradients at start
- The gradient of BCE + sigmoid simplifies to $(\hat{y} - y)$ — elegant and efficient